## Dropping Rows & Columns

* Why does this matter?
Real world datasets are messy. You'll always have columns you don't need (like ID, timestamp, irrelevant metadata), and rows that are garbage (duplicates, test entries, corrupted records). Keeping them pollutes your analysis. Dropping is one of the first things you do in any data cleaning task.
* The core method — drop()
* What is it?
drop() is the primary Pandas method to remove rows or columns from a DataFrame.

* Why one method for both rows and columns?

Because Pandas uses the axis parameter to distinguish — *axis=0* means rows,

 *axis=1* means columns. Same method, different direction. You already know 
 
 *axis* from *apply()* — same concept here.
* How does it work?

You tell Pandas what labels to drop, and which axis to look along. Pandas finds those labels and removes them, returning a new DataFrame

In [6]:
import pandas as pd

data = {
    'Name':    ['Sumed', 'Priya', 'Rahul', 'Neha', 'Arjun'],
    'Dept':    ['Data', 'HR', 'Data', 'Finance', 'Data'],
    'Salary':  [60000, 45000, 75000, 55000, 80000],
    'Exp':     [3, 2, 5, 4, 6],
    'Temp_ID': ['T1', 'T2', 'T3', 'T4', 'T5']  # irrelevant column
}
df = pd.DataFrame(data)

In [7]:
df

,Name,Dept,Salary,Exp,Temp_ID
0,Sumed,Data,60000,3,T1
1,Priya,HR,45000,2,T2
2,Rahul,Data,75000,5,T3
3,Neha,Finance,55000,4,T4
4,Arjun,Data,80000,6,T5


## Drop a single column


In [8]:
# drop() syntax: df.drop(labels, axis)
# axis=1 means we are working along the column direction
# axis=0 means we are working along the row direction

# Drop single column — returns new df, original unchanged
df.drop('Temp_ID', axis=1)

# Cleaner alternative using 'columns' keyword
# No need to write axis=1 — more readable
df.drop(columns='Temp_ID')

# To actually save the result, reassign
df = df.drop(columns='Temp_ID')

In [4]:
df

,Name,Dept,Salary,Exp
0,Sumed,Data,60000,3
1,Priya,HR,45000,2
2,Rahul,Data,75000,5
3,Neha,Finance,55000,4
4,Arjun,Data,80000,6


## Drop multiple column

In [ ]:
# Pass a LIST of column names to drop multiple at once
# Why a list? Because drop() expects an iterable of labels

df = df.drop(columns=['Temp_ID', 'Exp'])

# Real world use case:
# After EDA you identify irrelevant columns → drop them all at once
irrelevant_cols = ['Temp_ID', 'Exp']
df = df.drop(columns=irrelevant_cols)  # clean, readable

## Drop rows by index label

In [10]:
# axis=0 (default) → drop operates on rows
# Pass the index LABEL (not position) of the row to drop

# Drop single row with index label 2
df.drop(2)               # axis=0 is default, no need to write it
df.drop(index=2)         # cleaner with 'index' keyword



,Name,Dept,Salary,Exp
0,Sumed,Data,60000,3
1,Priya,HR,45000,2
3,Neha,Finance,55000,4
4,Arjun,Data,80000,6


In [11]:
# Drop multiple rows
df.drop(index=[0, 3])     # drop rows 0 and 3

# WHY by label and not position?
# drop() always uses INDEX LABELS — not integer positions
# If your index is [10, 20, 30], you write drop(index=10)
# NOT drop(index=0) — that would raise KeyError

,Name,Dept,Salary,Exp
1,Priya,HR,45000,2
2,Rahul,Data,75000,5
4,Arjun,Data,80000,6


 Notice index jumps from 1 → 3. 

Row 2 is gone but the remaining index labels don't reset automatically. 

Use .reset_index(drop=True) after dropping if you want clean 0,1,2,3... index.

## Reset index after droping rows

In [13]:
# After dropping rows, index has gaps: 0, 1, 3, 4
# reset_index() resets it back to clean 0, 1, 2, 3

df_dropped = df.drop(index=2)
print(df_dropped.index.tolist())   # [0, 1, 3, 4] — gap at 2



[0, 1, 3, 4]


In [18]:
df_dropped

,Name,Dept,Salary,Exp
0,Sumed,Data,60000,3
1,Priya,HR,45000,2
3,Neha,Finance,55000,4
4,Arjun,Data,80000,6


In [14]:
df_clean = df_dropped.reset_index(drop=True)
print(df_clean.index.tolist())      # [0, 1, 2, 3] — clean

# WHY drop=True inside reset_index?
# Without drop=True, the old index gets added as a new column
# drop=True discards the old index completely

[0, 1, 2, 3]


In [17]:
df_clean

,Name,Dept,Salary,Exp
0,Sumed,Data,60000,3
1,Priya,HR,45000,2
2,Neha,Finance,55000,4
3,Arjun,Data,80000,6


## Drop rows based on a condition

In [20]:
# You almost NEVER drop rows by index number in real work
# You drop rows that MEET a condition
# The pattern: keep rows that DON'T match the bad condition
# i.e. filter WITH the opposite condition using boolean mask

# Drop all rows where Salary < 50000
df = df[df['Salary'] >= 50000]



In [21]:
# Drop rows where Dept is HR
df = df[df['Dept'] != 'HR']



In [22]:
# Drop rows where Name is missing (NaN)
df = df[df['Name'].notna()]

# WHY this pattern instead of drop()?
# drop() requires you to know the exact index labels
# Boolean filtering works on VALUES — more practical

In [23]:
df

,Name,Dept,Salary,Exp
0,Sumed,Data,60000,3
2,Rahul,Data,75000,5
3,Neha,Finance,55000,4
4,Arjun,Data,80000,6


This is the most common row-dropping pattern in real projects. 

You're not really "dropping" — you're keeping only what you want using a filter. 

Same result, cleaner intent.

## ERROR = "IGNORE"

In [ ]:
# If you try to drop a column that doesn't exist → KeyError
df.drop(columns='NonExistentCol')
# KeyError: "['NonExistentCol'] not found in axis"

# Fix: use errors='ignore' — silently skips if column not found
# Very useful in reusable scripts or pipelines
# where you're not sure if a column exists
df.drop(columns='NonExistentCol', errors='ignore')
# No error — just returns df as-is

# Real world use: cleaning pipeline runs on multiple datasets
# Not all datasets have the same columns — errors='ignore' is safe
cols_to_remove = ['Temp_ID', 'Debug_Flag', 'Internal_Note']
df = df.drop(columns=cols_to_remove, errors='ignore')

## keep only specific column(reverse drop)

In [25]:
# When you have 50 columns and want to keep only 3
# Dropping 47 columns is painful
# Better: just SELECT the columns you want to keep

# Instead of dropping 47 columns:
df = df[['Name', 'Salary', 'Dept']]

# This is NOT drop() — it's column selection
# But the END RESULT is same: unwanted columns are gone
# Rule of thumb:
# Few columns to remove → use drop()
# Few columns to KEEP → use column selection df[[...]]

In [26]:
df

,Name,Salary,Dept
0,Sumed,60000,Data
2,Rahul,75000,Data
3,Neha,55000,Finance
4,Arjun,80000,Data


 *In real EDA workflows, senior analysts almost always use column selection df[[cols]] over drop() — it's more intentional and self-documenting.*
